SECTION 1

In [ ]:
import pandas as pd
import numpy as np

from tqdm import tqdm
import random

SECTION 2
Sampling Configuration

(Define sample size and reproducibility)

In [ ]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
TARGET_PER_FILE = 500_000

SECTION 3
Reservoir Sampling Function

(Sample ratings without loading entire file)

In [ ]:
def sample_netflix_file(
    filename,
    target_size
):

    reservoir = []

    movie_id = None

    seen = 0

    with open(
        filename,
        "r"
    ) as f:

        for line in tqdm(
            f,
            desc=filename
        ):

            line = line.strip()

            if line.endswith(":"):

                movie_id = int(
                    line[:-1]
                )

            else:

                user_id, rating, date = (
                    line.split(",")
                )

                row = [
                    int(user_id),
                    movie_id,
                    int(rating),
                    date
                ]

                seen += 1

                if len(reservoir) < target_size:

                    reservoir.append(row)

                else:

                    j = random.randint(
                        1,
                        seen
                    )

                    if j <= target_size:

                        reservoir[
                            j - 1
                        ] = row

    return reservoir

SECTION 4
Sample All Files

(Create representative sample from entire Netflix catalog)

In [ ]:
files = [
    "combined_data_1.txt",
    "combined_data_2.txt",
    "combined_data_3.txt",
    "combined_data_4.txt"
]

In [ ]:
all_samples = []

for file in files:

    sample = sample_netflix_file(
        file,
        TARGET_PER_FILE
    )

    all_samples.extend(
        sample
    )

combined_data_1.txt: 24058263it [01:10, 342967.65it/s]
combined_data_2.txt: 26982302it [01:07, 399381.48it/s]
combined_data_3.txt: 22605786it [00:54, 414660.24it/s]
combined_data_4.txt: 26851926it [01:01, 439883.07it/s]


SECTION 5
Create Dataset

(Convert sampled ratings to DataFrame)

In [ ]:
ratings_df = pd.DataFrame(
    all_samples,
    columns=[
        "user_id",
        "movie_id",
        "rating",
        "date"
    ]
)

In [ ]:
ratings_df["date"] = pd.to_datetime(
    ratings_df["date"]
)

In [ ]:
ratings_df.head()

,user_id,movie_id,rating,date
0,1124982,2209,4,2005-03-13
1,1767452,1530,4,2005-11-03
2,2022945,4256,3,2005-09-02
3,558749,3463,5,2002-04-05
4,2507671,357,4,2005-03-09


SECTION 6
Dataset Statistics

(Verify sample quality)

In [ ]:
print(
    "Ratings:",
    len(ratings_df)
)

print(
    "Users:",
    ratings_df["user_id"].nunique()
)

print(
    "Movies:",
    ratings_df["movie_id"].nunique()
)

Ratings: 2000000
Users: 358375
Movies: 17324


In [ ]:
ratings_df["rating"].describe()

,rating
count,2.000000e+06
mean,3.605133e+00
std,1.085041e+00
min,1.000000e+00
25%,3.000000e+00
50%,4.000000e+00
75%,4.000000e+00
max,5.000000e+00


In [ ]:
print(
    ratings_df["date"].min()
)

print(
    ratings_df["date"].max()
)

1999-11-11 00:00:00
2005-12-31 00:00:00


SECTION 7
Coverage Analysis

(Check catalog representation)

In [ ]:
movie_coverage = (
    ratings_df["movie_id"]
    .nunique()
)

print(
    "Movie Coverage:",
    movie_coverage
)

Movie Coverage: 17324


In [ ]:
user_coverage = (
    ratings_df["user_id"]
    .nunique()
)

print(
    "User Coverage:",
    user_coverage
)

User Coverage: 358375


SECTION 8
Save Dataset

(Freeze modeling dataset for all future experiments)

In [ ]:
ratings_df.to_csv(
    "netflix_modeling_data.csv",
    index=False
)

In [ ]:
print(
    "Dataset Saved Successfully"
)

Dataset Saved Successfully


SECTION 9
Dataset Memory

(Estimate storage requirements)

In [ ]:
memory_mb = (
    ratings_df.memory_usage(
        deep=True
    ).sum()
    / 1024**2
)

print(
    f"Memory Usage: {memory_mb:.2f} MB"
)

Memory Usage: 61.04 MB
